# Phase 5 — Grade ablation conditions + compute score deltas (local, no Colab)

See `docs/research_proposal.md` §4.6. Runs entirely locally, like `grade_local.ipynb` -- no GPU/model needed. Grades all 4 ablation conditions x 2 organisms against the same rubric as Phase 3, then computes each condition's score delta vs. the unablated Phase 3 baseline. Produces `labels_{condition}_{organism}.jsonl` (8 files) and `phase5_ablation_results.json`.

In [1]:
%pip install -q anthropic pillow python-dotenv tqdm

Note: you may need to restart the kernel to use updated packages.


## Prerequisites checklist

- `artifacts/phase5_generations_A.json` and `phase5_generations_B.json` downloaded from Drive into local `artifacts/` (produced by `04_cross_ablation.ipynb` in Colab).
- `artifacts/labels_text_{A,B}.jsonl` and `labels_mm_{A,B}.jsonl` already local from `grade_local.ipynb` -- these are the Phase 3 baseline scores the deltas are computed against.
- `.env` with `ANTHROPIC_API_KEY` already set from before.

In [2]:
import os

from dotenv import load_dotenv

os.chdir('/Users/yelyzavetahusieva/Desktop/emergent-misalignment-project')
load_dotenv(dotenv_path='.env')  # explicit path -- find_dotenv()'s auto-detection can fail depending on execution context
print('ANTHROPIC_API_KEY set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

ANTHROPIC_API_KEY set: True


## Grade all 4 conditions, both organisms

Each ablated completion is compared against the *same* base-model completion used in Phase 3 (via `build_labeling_examples`), so scores land on the same scale and the delta against the Phase 3 unablated score is meaningful. Images only apply to the two `*_mm` conditions.

In [5]:
import json
from pathlib import Path

from src.generate import build_labeling_examples, load_multimodal_eval_set
from src.judge import grade_batch_api

ARTIFACTS = Path('artifacts')
mm_images = [ex['image'] for ex in load_multimodal_eval_set()]

for organism in ['A', 'B']:
    generations = json.loads((ARTIFACTS / f'phase5_generations_{organism}.json').read_text())

    for name, data in generations.items():
        images = mm_images if 'mm' in name else None
        examples = build_labeling_examples(
            data['prompts'], data['base_completions'], data['ablated_completions'],
            images=images, id_prefix=f'{name}_',
        )
        grade_batch_api(examples, out_path=ARTIFACTS / f'labels_{name}_{organism}.jsonl')

    print(f'organism {organism}: grading done')

grading labels_within_text_A.jsonl: 100%|██████████| 30/30 [00:00<?, ?it/s]


organism A: grading done


grading labels_within_text_B.jsonl: 100%|██████████| 30/30 [06:06<00:00, 12.20s/it]

organism B: grading done


## Compute score deltas vs. the unablated Phase 3 scores, per organism

Positive delta = ablation reduced misalignment. `cross_mm`/`cross_text` are the tests that matter; `within_mm`/`within_text` are the baselines they're compared against (does the cross-modal ablation approach the within-modal ablation's effect?).

In [6]:
from src.judge import load_scores

def condition_deltas(name: str, organism: str, baseline_scores: dict, modality_prefix: str) -> list[int]:
    ablated_scores = load_scores(ARTIFACTS / f'labels_{name}_{organism}.jsonl')
    deltas = []
    for ablated_id, ablated_score in ablated_scores.items():
        suffix = ablated_id[len(name) + 1:]  # strip '{name}_' prefix -> 'p{i}_s{j}'
        baseline_id = f'{modality_prefix}_{suffix}'
        if baseline_id in baseline_scores:
            deltas.append(baseline_scores[baseline_id] - ablated_score)  # positive = ablation reduced misalignment
    return deltas

results = {}
for organism in ['A', 'B']:
    text_scores = load_scores(ARTIFACTS / f'labels_text_{organism}.jsonl')
    mm_scores = load_scores(ARTIFACTS / f'labels_mm_{organism}.jsonl')
    results[organism] = {
        'cross_mm': condition_deltas('cross_mm', organism, mm_scores, 'mm'),
        'within_mm': condition_deltas('within_mm', organism, mm_scores, 'mm'),
        'cross_text': condition_deltas('cross_text', organism, text_scores, 'text'),
        'within_text': condition_deltas('within_text', organism, text_scores, 'text'),
    }

for organism, conds in results.items():
    print(f'--- organism {organism} ---')
    for name, deltas in conds.items():
        mean_delta = sum(deltas) / len(deltas) if deltas else float('nan')
        print(f'  {name}: n={len(deltas)}, mean delta={mean_delta:.1f}')

print('\nKey comparison per organism: does cross_mm delta approach within_mm delta? (and cross_text vs within_text?)')
print('Key comparison across organisms: does the pattern in A also hold in B? See proposal Table 5.2.')

(ARTIFACTS / 'phase5_ablation_results.json').write_text(json.dumps(results))
print('\nsaved phase5_ablation_results.json')

--- organism A ---
  cross_mm: n=30, mean delta=5.5
  within_mm: n=30, mean delta=7.8
  cross_text: n=30, mean delta=14.0
  within_text: n=30, mean delta=14.6
--- organism B ---
  cross_mm: n=30, mean delta=-6.9
  within_mm: n=30, mean delta=-6.6
  cross_text: n=30, mean delta=-9.4
  within_text: n=30, mean delta=-9.1

Key comparison per organism: does cross_mm delta approach within_mm delta? (and cross_text vs within_text?)
Key comparison across organisms: does the pattern in A also hold in B? See proposal Table 5.2.

saved phase5_ablation_results.json


## Next step

`phase5_ablation_results.json` and the 8 `labels_*.jsonl` files are already local -- also make sure `phase4_direction_summary.json` (from `03_extract_directions.ipynb`, still Drive-only so far) is downloaded to local `artifacts/`. Then `05_analysis.ipynb` (already local-only) can run without touching Colab again.